In [ ]:
"""
TRANSCORBE INTERSTATION CCF COMPUTATION PIPELINE
Generates CCFs for dispersion picking & Moho depth migration
"""

import numpy as np
import scipy.signal as signal
from scipy.signal import hilbert
from scipy import fftpack
from obspy import read, read_inventory
from obspy.geodetics import gps2dist_azimuth
import glob
import os
from tqdm import tqdm
import time
import json

# ==============================================================================
# CONFIG
# ==============================================================================

ROOT_PATH = r"D:\Alan CUSAT- dissertation\TRANSCORBE\9H"
OUTPUT_ROOT = r"D:\Alan CUSAT- dissertation\TRANSCORBE\CCF_DISPERSION"
STATION_XML = r"D:\Alan CUSAT- dissertation\TRANSCORBE\stations.xml"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

TARGET_DT = 0.2
FREQMIN = 0.05
FREQMAX = 1.0
MAXLAG_SEC = 150
MIN_OVERLAP_HOURS = 12
MIN_SAMPLES = 1000

# ==============================================================================
# LOAD STATION COORDS
# ==============================================================================

COORDS = {}

try:
    inv = read_inventory(STATION_XML)
    for net in inv:
        for sta in net:
            COORDS[sta.code] = {
                "lat": sta.latitude,
                "lon": sta.longitude
            }
    print(f"Loaded {len(COORDS)} stations from {STATION_XML}\n")
except Exception as e:
    print(f"WARNING: Could not load {STATION_XML}: {e}")
    print("Will use all adjacent pairs\n")

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def comb_filter(trace, fs):
    """Envelope-based adaptive weighting normalization"""
    p1 = [0.5, 2, 6]
    p2 = [2, 6, 20]
    nyq = fs / 2
    weights = []
    
    for T1, T2 in zip(p1, p2):
        fmax = 1.0 / T1
        fmin = 1.0 / T2
        if fmax >= nyq:
            continue
        b, a = signal.butter(2, [fmin, fmax], btype="band", fs=fs)
        tr = signal.filtfilt(b, a, trace)
        env = np.abs(hilbert(tr))
        weights.append(1.0 / (env + 1e-10))
    
    if not weights:
        return trace / (np.max(np.abs(trace)) + 1e-10)
    
    W = np.ones_like(trace)
    for w in weights:
        W *= w
    
    water_level = 0.10 * np.mean(W)
    W[W < water_level] = water_level
    trace_norm = trace * W
    
    return trace_norm / (np.max(np.abs(trace_norm)) + 1e-10)

def crosscorrelation(trace01, trace02, maxlag):
    """FFT-based cross-correlation"""
    lentrace = len(trace01)
    goodnumber = int(maxlag + lentrace)
    
    tr2 = np.zeros(goodnumber)
    tr2[0:lentrace] = trace02
    tr2[0:lentrace] /= np.sqrt(np.sum(tr2[0:lentrace] ** 2))
    tr2 = fftpack.fft(tr2, overwrite_x=True)
    tr2.imag *= -1
    
    tr1 = np.zeros(goodnumber)
    tr1[maxlag:maxlag+lentrace] = trace01
    tr1[maxlag:maxlag+lentrace] /= np.sqrt(
        np.sum(tr1[maxlag:maxlag+lentrace] ** 2))
    
    tr2 *= fftpack.fft(tr1, overwrite_x=True)
    
    return fftpack.ifft(tr2, overwrite_x=True)[0:2*maxlag+1].real.astype(np.float32)

def symmetrize_ccf(ccf):
    """Symmetrize: (causal + acausal)/2"""
    n = len(ccf)
    center = n // 2
    causal = ccf[center:]
    acausal = ccf[:center+1][::-1]
    min_len = min(len(causal), len(acausal))
    sym = 0.5 * (causal[:min_len] + acausal[:min_len])
    return sym

def preprocess_trace(tr, fs):
    """Standard preprocessing: detrend, filter, taper, comb-filter"""
    if abs(tr.stats.delta - TARGET_DT) > 1e-5:
        tr.resample(1 / TARGET_DT)
    
    fs = tr.stats.sampling_rate
    
    tr.detrend("linear")
    tr.detrend("demean")
    tr.filter("bandpass", freqmin=FREQMIN, freqmax=FREQMAX, zerophase=True)
    tr.taper(0.05)
    
    raw = tr.data.astype(np.float32)
    norm = comb_filter(raw, fs)
    
    return norm, fs

def safe_read(filepath):
    """Safely read mseed file"""
    try:
        st = read(filepath)
        if len(st) == 0:
            return None
        return st[0]
    except:
        return None

# ==============================================================================
# FIND STATIONS
# ==============================================================================

# Look for TR* station folders inside year directories
stations = set()

for year_dir in glob.glob(os.path.join(ROOT_PATH, "20*")):
    for station_folder in glob.glob(os.path.join(year_dir, "TR*")):
        station_name = os.path.basename(station_folder)
        stations.add(station_name)

stations = sorted(list(stations))

print(f"Found {len(stations)} stations:")
for s in stations:
    print(f"  {s}")
print()

# ==============================================================================
# BUILD STATION PAIR LIST
# ==============================================================================

station_pairs = []

if COORDS:
    # Use all pairs within distance range
    MIN_DIST = 15  # km
    MAX_DIST = 60  # km
    
    for i in range(len(stations)):
        for j in range(i+1, len(stations)):
            s1, s2 = stations[i], stations[j]
            if s1 in COORDS and s2 in COORDS:
                lat1, lon1 = COORDS[s1]["lat"], COORDS[s1]["lon"]
                lat2, lon2 = COORDS[s2]["lat"], COORDS[s2]["lon"]
                dist_m, _, _ = gps2dist_azimuth(lat1, lon1, lat2, lon2)
                dist_km = dist_m / 1000.0
                if MIN_DIST <= dist_km <= MAX_DIST:
                    station_pairs.append((s1, s2, dist_km))
else:
    # Adjacent pairs only
    for i in range(len(stations)-1):
        station_pairs.append((stations[i], stations[i+1], -1))

print(f"Processing {len(station_pairs)} station pairs\n")

# ==============================================================================
# INDEX FILES BY STATION AND DATE
# ==============================================================================

print("Indexing data files...")

file_db = {s: {} for s in stations}

for station in stations:
    for year_dir in glob.glob(os.path.join(ROOT_PATH, "20*")):
        station_path = os.path.join(year_dir, station, "HHZ")
        
        if not os.path.exists(station_path):
            continue
        
        files = glob.glob(os.path.join(station_path, "*.mseed"))
        
        for f in files:
            fname = os.path.basename(f)
            try:
                # Extract date from filename (assuming format: STATION_YYYY-MM-DD...)
                parts = fname.split("_")
                if len(parts) >= 2:
                    date_str = parts[1].split(".")[0]  # Get YYYY-MM-DD
                    file_db[station][date_str] = f
            except:
                continue

# Count stations with data
stations_with_data = [s for s in file_db if file_db[s]]
print(f"Found data for {len(stations_with_data)} stations")
print(f"Total files indexed: {sum(len(file_db[s]) for s in stations_with_data)}\n")

# ==============================================================================
# MAIN PROCESSING LOOP
# ==============================================================================

print("="*80)
print("COMPUTING CCFs")
print("="*80 + "\n")

start_total = time.time()
processed_pairs = 0
total_jobs = 0

pbar = tqdm(station_pairs, desc="Pairs", ncols=120, ascii=False)

for sta1, sta2, dist in pbar:
    
    # Find common dates
    dates1 = set(file_db[sta1].keys())
    dates2 = set(file_db[sta2].keys())
    common_dates = sorted(list(dates1 & dates2))
    
    if not common_dates:
        continue
    
    # Create output directory
    outdir = os.path.join(OUTPUT_ROOT, f"{sta1}_{sta2}")
    os.makedirs(outdir, exist_ok=True)
    
    daily_ccfs = []
    valid_days = 0
    
    # Process each common day
    for date_str in common_dates:
        
        try:
            file1 = file_db[sta1][date_str]
            file2 = file_db[sta2][date_str]
            
            # Output file
            outfile = os.path.join(outdir, f"{date_str}_ccf.npy")
            
            # Skip if already processed
            if os.path.exists(outfile):
                continue
            
            # Read traces
            tr1 = safe_read(file1)
            tr2 = safe_read(file2)
            
            if tr1 is None or tr2 is None:
                continue
            
            if len(tr1.data) < MIN_SAMPLES or len(tr2.data) < MIN_SAMPLES:
                continue
            
            # Preprocess
            norm1, fs = preprocess_trace(tr1, tr1.stats.sampling_rate)
            norm2, fs = preprocess_trace(tr2, tr2.stats.sampling_rate)
            
            # Find overlap
            start = max(tr1.stats.starttime, tr2.stats.starttime)
            end = min(tr1.stats.endtime, tr2.stats.endtime)
            
            if start >= end:
                continue
            
            overlap_hours = (end - start) / 3600.0
            
            if overlap_hours < MIN_OVERLAP_HOURS:
                continue
            
            # Trim to common window
            idx_start1 = int((start - tr1.stats.starttime) * fs)
            idx_end1 = int((end - tr1.stats.starttime) * fs)
            
            idx_start2 = int((start - tr2.stats.starttime) * fs)
            idx_end2 = int((end - tr2.stats.starttime) * fs)
            
            norm1 = norm1[idx_start1:idx_end1]
            norm2 = norm2[idx_start2:idx_end2]
            
            nmin = min(len(norm1), len(norm2))
            norm1 = norm1[:nmin]
            norm2 = norm2[:nmin]
            
            # Compute CCF
            maxlag = int(MAXLAG_SEC * fs)
            ccf = crosscorrelation(norm1, norm2, maxlag)
            ccf = symmetrize_ccf(ccf)
            ccf /= (np.max(np.abs(ccf)) + 1e-10)
            
            # Save daily CCF
            np.save(outfile, ccf.astype(np.float32))
            valid_days += 1
            daily_ccfs.append(ccf)
            
        except Exception as e:
            continue
    
    # Stack daily CCFs if we have enough
    if len(daily_ccfs) >= 3:
        ccf_stack = np.mean(daily_ccfs, axis=0).astype(np.float32)
        stack_file = os.path.join(outdir, "STACK_ccf.npy")
        np.save(stack_file, ccf_stack)
        
        # Save metadata
        meta = {
            "station1": sta1,
            "station2": sta2,
            "distance_km": float(dist) if dist > 0 else -1,
            "num_days": valid_days,
            "maxlag_sec": MAXLAG_SEC,
            "fs": float(fs),
            "target_dt": TARGET_DT
        }
        
        meta_file = os.path.join(outdir, "metadata.json")
        with open(meta_file, 'w') as f:
            json.dump(meta, f, indent=2)
    
    processed_pairs += 1
    elapsed = time.time() - start_total
    
    if processed_pairs > 0:
        avg_time = elapsed / processed_pairs
        remaining = len(station_pairs) - processed_pairs
        eta_sec = remaining * avg_time
        pbar.set_postfix({
            "pair": f"{sta1}-{sta2}",
            "days": f"{valid_days}",
            "ETA(min)": f"{eta_sec/60:.1f}"
        })

pbar.close()

# ==============================================================================
# SUMMARY
# ==============================================================================

total_time = time.time() - start_total

print("\n" + "="*80)
print("DONE")
print("="*80)
print(f"Total time: {total_time/60:.1f} min")
print(f"Processed {processed_pairs} station pairs")
print(f"\nOutput: {OUTPUT_ROOT}")
print("\nEach pair folder contains:")
print("  - Daily CCF files (YYYY-MM-DD_ccf.npy)")
print("  - Stacked CCF (STACK_ccf.npy)")
print("  - Metadata (metadata.json)")
print("\nReady for FTAN dispersion picking!")